# Reading from the Bronze Layer

In [0]:
bronze_df = spark.table("workspace.bronze.crm_cust_info")

# Init

In [0]:
from pyspark.sql.types import StringType
from pyspark.sql.functions import col, trim, when

In [0]:
rename_map = {
    'cst_id': 'customer_id',
    'cst_key': 'customer_key',
    'cst_firstname': 'first_name',
    'cst_lastname': 'last_name',
    'cst_marital_status': 'marital_status',
    'cst_gndr': 'gender',
    'cst_create_date': 'created_date'
}


# Data Transformations

## Trimming whitespaces from text columns

In [0]:
for field in bronze_df.schema.fields:
    if isinstance(field.dataType, StringType):
        bronze_df = bronze_df.withColumn(field.name, trim(col(field.name)))

## Normalizing abbreviations to descriptive business values

In [0]:
bronze_df = (
    bronze_df
    
    .withColumn(
    "cst_marital_status",
    when(col("cst_marital_status") == "S", "Single")
    .when(col("cst_marital_status") == "M", "Married")
    .otherwise("n/a")
    )
   .withColumn(
       "cst_gndr",
       when(col("cst_gndr") == "M", "Male")
       .when(col("cst_gndr") == "F", 'Female')
       .otherwise('n/a')
   )
)

## Renaming cryptic columns using a translation dictionary

In [0]:
for old_name, new_name in rename_map.items():
    if old_name in bronze_df.columns:
        bronze_df = bronze_df.withColumnRenamed(old_name, new_name)
        

#Writing into Silver table

In [0]:
bronze_df.write \
    .mode("overwrite") \
    .format('delta') \
    .saveAsTable("silver.crm_customers")